# Mixed-P dataset


In [ ]:
'''
This script sets up and trains a GPT-style transformer model using Karpathy’s minGPT framework to perform bit-level prediction on synthetic binary sequences with varying probabilities.

Key Steps:
- Clones and installs the minGPT repository.
- Defines a custom dataset (`MixedPBinarySequenceDataset`) that generates binary sequences using a mix of Bernoulli distributions (P = 0.2, 0.5, 0.8).
- Initializes a small GPT model (`gpt-nano`) to predict the next bit in the sequence.
- Sets up a training loop using minGPT’s Trainer class, logs loss every 100 iterations, and trains for 2000 iterations.
- Saves the trained model weights for later use in entropy modeling or compression tasks.

The goal is to learn a probability distribution over binary sequences to support context-aware, entropy-efficient lossless compression.
'''


# Install and set up minGPT
!git clone https://github.com/karpathy/minGPT.git
%cd minGPT/
%pip install -e .

import torch
from torch.utils.data import Dataset
from mingpt.utils import set_seed
from mingpt.model import GPT
from mingpt.trainer import Trainer

import random

# Set reproducibility seed
set_seed(3407)

# Mixed-P Binary Sequence Dataset
class MixedPBinarySequenceDataset(Dataset):
    def __init__(self, length=128, num_sequences=10000, p_list=[0.2, 0.5, 0.8]):
        self.length = length
        self.num_sequences = num_sequences
        self.vocab_size = 2  # Binary
        self.p_list = p_list  # List of Bernoulli probabilities (for mixed patterns)
        # Assign a random P to each sequence
        self.sequence_ps = [random.choice(self.p_list) for _ in range(num_sequences)]

    def __len__(self):
        return self.num_sequences

    def get_vocab_size(self):
        return self.vocab_size

    def get_block_size(self):
        return self.length - 1  # Predict next bit

    def __getitem__(self, idx):
        p = self.sequence_ps[idx]
        sequence = torch.where(
            torch.rand(size=(self.length,)) < p,
            torch.tensor(1),
            torch.tensor(0)
        ).type(torch.long)

        x = sequence[:-1]
        y = sequence[1:]
        return x, y

# Initialize dataset
train_dataset = MixedPBinarySequenceDataset(length=128, num_sequences=10000, p_list=[0.2, 0.5, 0.8])

# Define GPT model
model_config = GPT.get_default_config()
model_config.model_type = 'gpt-nano'
model_config.vocab_size = train_dataset.get_vocab_size()
model_config.block_size = train_dataset.get_block_size()
model = GPT(model_config)

# Trainer setup
train_config = Trainer.get_default_config()
train_config.learning_rate = 5e-4
train_config.max_iters = 2000
train_config.num_workers = 0
trainer = Trainer(train_config, model, train_dataset)

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Progress callback
def batch_end_callback(trainer):
    if trainer.iter_num % 100 == 0:
        print(f"iter {trainer.iter_num}: train loss {trainer.loss.item():.5f}")
trainer.set_callback('on_batch_end', batch_end_callback)

# Run training
trainer.run()

# Save trained model weights
torch.save(model.state_dict(), 'model_weights.pth')


Cloning into 'minGPT'...
remote: Enumerating objects: 489, done.
remote: Counting objects: 100% (239/239), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 489 (delta 153), reused 145 (delta 145), pack-reused 250 (from 1)
Receiving objects: 100% (489/489), 1.43 MiB | 7.89 MiB/s, done.
Resolving deltas: 100% (268/268), done.
/content/minGPT
Obtaining file:///content/minGPT
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 101.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.3 MB/s eta 0:00:00
   ━━

In [ ]:
'''
This script evaluates a trained GPT-style transformer model's ability to predict the next bit in binary sequences.

Key Functions:
- `compute_probability(model, sequence, device)`: Takes a binary sequence and uses the model to compute the probability distribution over the next bit (0 or 1). Returns full softmax probabilities for inspection.

Usage:
- Tests the model on short (length-7) binary sequences generated from Bernoulli distributions with known probabilities (`P=0.1` to `0.8`).
- Prints both the test sequences and the model's predicted probability for the next bit.

Purpose:
This evaluation helps determine whether the model’s predictions match the underlying data distribution — a critical step in verifying its utility as an entropy model for compression.
'''

# Probability evaluator
def compute_probability(model, sequence, device):
    model.eval()
    sequence = torch.tensor(sequence, dtype=torch.long).unsqueeze(0).to(device)
    with torch.no_grad():
        logits, _ = model(sequence)
        probs = torch.nn.functional.softmax(logits, dim=-1)
        next_bit_probs = probs[0, -1]
        next_bit_prob = next_bit_probs[1].item()
        return probs

# Test evaluation on known P sequences
test_ps = [0.1, 0.15, 0.2, 0.25, 0.33, 0.5, 0.7, 0.75, 0.8]
for p in test_ps:
    print(f"\n--- Testing on P = {p} ---")
    # Generate a test sequence
    test_sequence = torch.where(
        torch.rand(size=(7,)) < p,
        torch.tensor(1),
        torch.tensor(0)
    ).tolist()
    print(f"Test sequence (length 7): {test_sequence}")
    predicted_prob = compute_probability(model, test_sequence, device)
    print(predicted_prob[0,:,-1])
    #print(f"Predicted probability next bit is 1: {predicted_prob:.4f} (true P = {p})")


--- Testing on P = 0.1 ---
Test sequence (length 7): [0, 0, 0, 0, 0, 0, 0]
tensor([0.3717, 0.3163, 0.2660, 0.2387, 0.2259, 0.2164, 0.2097])

--- Testing on P = 0.15 ---
Test sequence (length 7): [0, 1, 0, 0, 0, 0, 0]
tensor([0.3717, 0.5051, 0.4106, 0.3388, 0.2905, 0.2592, 0.2395])

--- Testing on P = 0.2 ---
Test sequence (length 7): [0, 0, 0, 0, 1, 0, 1]
tensor([0.3717, 0.3163, 0.2660, 0.2387, 0.2857, 0.2475, 0.3369])

--- Testing on P = 0.25 ---
Test sequence (length 7): [1, 0, 0, 0, 1, 0, 0]
tensor([0.6489, 0.5114, 0.4130, 0.3258, 0.4390, 0.3515, 0.3025])

--- Testing on P = 0.33 ---
Test sequence (length 7): [1, 0, 0, 0, 0, 0, 0]
tensor([0.6489, 0.5114, 0.4130, 0.3258, 0.2822, 0.2511, 0.2322])

--- Testing on P = 0.5 ---
Test sequence (length 7): [1, 1, 1, 0, 0, 0, 1]
tensor([0.6489, 0.6839, 0.7148, 0.6446, 0.5590, 0.4994, 0.5597])

--- Testing on P = 0.7 ---
Test sequence (length 7): [0, 1, 1, 1, 1, 1, 1]
tensor([0.3717, 0.5051, 0.5759, 0.6521, 0.6873, 0.7215, 0.7477])

--- Testi

In [ ]:
'''
This line prints the model’s predicted probability distribution over the next bit (0 or 1) given a sequence.

Details:
- `predicted_prob` is the softmax output from the transformer model.
- `predicted_prob[0, :, -1]` accesses the logits for the last token in the sequence (i.e., the next bit prediction).
- The output is a tensor of two values: [P(bit=0), P(bit=1)].

Purpose:
Useful for verifying whether the model is assigning appropriate probabilities based on the input sequence — a key component for entropy coding.
'''
# sequential probability
print(predicted_prob[0,:,-1])

tensor([0.6489, 0.6839, 0.7148, 0.6446, 0.6881, 0.7187, 0.6574])


# Comparison to Universal

In [ ]:
'''
This script computes the average test loss (in bits per symbol) for a trained transformer model, providing a quantitative benchmark against the universal source coding entropy rate.

Function:
- `compute_test_loss(test_sequences, predicted_probs)`:
  Compares the model's predicted probabilities to the actual bits in test sequences using binary cross-entropy (log loss).
  It shifts the sequence to align predictions with the true "next bit" values.

Key Concepts:
- Implements binary cross-entropy loss: a proxy for how many bits the model uses, on average, to encode a symbol.
- Clips predicted probabilities to avoid log(0) for numerical stability.
- Returns the average loss across all bits in all sequences.

Purpose:
Used to evaluate how close the model's compression performance comes to the entropy limit defined by universal source coding theory.
'''
import numpy as np

def compute_test_loss(test_sequences, predicted_probs):
    total_loss = 0
    total_count = 0

    for seq, probs in zip(test_sequences, predicted_probs):
        # Shift sequence: predict each bit from previous
        true_next_bits = seq[1:]  # true next bits
        pred_probs = probs[:-1]   # predicted probs (aligned)

        for true_bit, pred_prob in zip(true_next_bits, pred_probs):
            # Clip to avoid log(0)
            pred_prob = np.clip(pred_prob, 1e-12, 1 - 1e-12)
            # Binary log loss
            loss = - (true_bit * np.log(pred_prob) + (1 - true_bit) * np.log(1 - pred_prob))
            total_loss += loss
            total_count += 1

    avg_loss = total_loss / total_count
    return avg_loss

In [ ]:
'''
This script evaluates the model's compression performance using hardcoded test sequences and their corresponding predicted probabilities.

Key Elements:
- `test_sequences`: A set of short, manually defined binary sequences meant to reflect different bit patterns and structures.
- `predicted_probs`: Precomputed probability values output by the model for each bit in each test sequence.

Process:
- Uses `compute_test_loss()` to calculate the average binary cross-entropy loss across all sequences.
- Prints the average log-loss, which corresponds to the number of bits per symbol (bps) the model would theoretically use to compress the data.

Comparison Guidance:
- This test loss can be directly compared to the entropy rate of the sequences or to the output of standard compressors (e.g., gzip, LZMA).
- For compressor comparison: compress the same sequences as a file, measure the compressed size in bits, and divide by the number of original bits to compute the effective bits per symbol.

Purpose:
Provides a grounded, interpretable benchmark for assessing whether the model-based compression is competitive with traditional universal compression methods.
'''
# Test sequences and predicted probabilities imported from probability evaluator
test_sequences = [
    [1, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0, 1],
    [0, 0, 1, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [1, 1, 1, 1, 1, 0, 1],
    [0, 1, 1, 1, 0, 1, 0],
    [1, 0, 1, 1, 0, 1, 1],
    [1, 0, 1, 1, 1, 0, 1],
    [1, 1, 1, 1, 1, 0, 0],
    # add all your test sequences here
]

predicted_probs = [
    [0.6489, 0.5114, 0.4130, 0.3258, 0.2822, 0.2511, 0.2322],
    [0.3717, 0.3163, 0.4005, 0.3296, 0.2833, 0.2548, 0.3374],
    [0.3717, 0.3163, 0.4005, 0.3296, 0.2833, 0.3694, 0.3073],
    [0.3717, 0.3163, 0.2660, 0.2387, 0.2259, 0.2164, 0.2097],
    [0.6489, 0.6839, 0.7148, 0.7445, 0.7559, 0.7222, 0.7520],
    [0.3717, 0.5051, 0.5759, 0.6521, 0.5601, 0.6278, 0.5585],
    [0.6489, 0.5114, 0.5887, 0.6516, 0.5578, 0.6221, 0.6694],
    [0.6489, 0.5114, 0.5887, 0.6516, 0.6896, 0.6089, 0.6727],
    [0.6489, 0.6839, 0.7148, 0.7445, 0.7559, 0.7222, 0.6627],
    # add all your predicted prob lists here
]

test_loss = compute_test_loss(test_sequences, predicted_probs)
print(f"Average test log-loss: {test_loss:.4f} bits per bit")

# To compare this value to universal source coding, encode same sequences using universal compressor
# such as gzip, bzip2, LZMA. Measure compressed file size and divide by number of original bits to
# get the bits per bit/symbol.

# Number of original bits is found by simply finding the length of the concatenated test sequences

Average test log-loss: 0.6403 bits per bit


In [ ]:
'''
This script computes the theoretical universal source coding (USC) efficiency for a set of binary test sequences, based on the entropy estimation formula from *Elements of Information Theory, 2nd Edition* (Cover & Thomas).

Key Components:
- `binary_entropy(p)`: Calculates binary entropy for a Bernoulli process with probability p.
- For each sequence:
  - Computes the empirical bit probability `p̂` (fraction of 1s),
  - Calculates its entropy H(p̂),
  - Adds the universal coding penalty term `(1/2) * log2(n) / n` to get the USC bound.

Purpose:
- The result estimates the minimum achievable average bits per symbol when using a universal (non-source-aware) compressor.
- Serves as a baseline for comparing model-based compression: if the transformer's BPS approaches the USC value, it's nearing optimal efficiency for unknown sources.

Output:
- Prints p̂, the entropy H, and the USC-adjusted rate for each sequence.
'''
import numpy as np

# Test sequences
test_sequences = [
    [1, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0, 1],
    [0, 0, 1, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [1, 1, 1, 1, 1, 0, 1],
    [0, 1, 1, 1, 0, 1, 0],
    [1, 0, 1, 1, 0, 1, 1],
    [1, 0, 1, 1, 1, 0, 1],
    [1, 1, 1, 1, 1, 0, 0],
]

def binary_entropy(p):
    if p == 0 or p == 1:
        return 0.0
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)

n = len(test_sequences[0])
log_n_over_n = 0.5 * np.log2(n) / n

for i, seq in enumerate(test_sequences):
    k = sum(seq)
    p_hat = k / n
    H = binary_entropy(p_hat)
    USC = H + log_n_over_n
    print(f"Seq {i+1}: p̂={p_hat:.2f}, Entropy={H:.4f}, USC={USC:.4f}")


Seq 1: p̂=0.14, Entropy=0.5917, USC=0.7922
Seq 2: p̂=0.29, Entropy=0.8631, USC=1.0636
Seq 3: p̂=0.29, Entropy=0.8631, USC=1.0636
Seq 4: p̂=0.00, Entropy=0.0000, USC=0.2005
Seq 5: p̂=0.86, Entropy=0.5917, USC=0.7922
Seq 6: p̂=0.57, Entropy=0.9852, USC=1.1858
Seq 7: p̂=0.71, Entropy=0.8631, USC=1.0636
Seq 8: p̂=0.71, Entropy=0.8631, USC=1.0636
Seq 9: p̂=0.71, Entropy=0.8631, USC=1.0636
